# Data Preparation Workflow
- adapté au dataset de films suivant : [IMDB Dataset of Top 1000 Movies and TV Shows](https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows)

### Libraries

In [321]:
import time
start = time.time()

In [322]:
# !pip install pandas scikit-learn matplotlib
# !pip install numpy

In [ ]:
import os, re, json
import unicodedata
import pandas as pd
import numpy as np 
import unicodedata
import html
import re
from sklearn.preprocessing import LabelEncoder

In [324]:
label_encoder = LabelEncoder()

### Outils

In [325]:
def uniformisation_string_to_int_encoding(dataframe, column: str, new_column_name: str):
    dataframe.loc[:, column] = dataframe[column].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)
    dataframe.loc[:, new_column_name] = label_encoder.fit_transform(dataframe[column].astype(str))
    
    return dataframe

In [326]:
def get_outlier_by_iqr(dataframe, column: str) -> pd.DataFrame:
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    
    lower_bound = q1 - 1.5 * (q3 - q1)
    upper_bound = q3 + 1.5 * (q3 - q1)
    
    outliers = dataframe[(dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)]
    return outliers

# test
# df = pd.DataFrame({'A': [1, 10000, 3, 4, 5]})
# pd_outliers = get_outlier_by_iqr(df, 'A')
# print(pd_outliers)

### Chargement des données

- charger les données en DataFrame

In [327]:
path = input("Entrer le chemin du dataset CSV: ")
    
# vérifier l'existence du dossier ../data et valider le chemin du fichier CSV
default_dir = "../data"
default_path = os.path.join(default_dir, "top_movies.csv")

os.makedirs(default_dir, exist_ok=True)

# nettoyer la saisie utilisateur
path = (path or "").strip()

# si vide ou extension invalide -> utiliser le fichier par défaut
if not path or not path.lower().endswith(".csv"):
    path = default_path

# si le chemin choisi n'existe pas, tenter le fallback sur le fichier par défaut
if not os.path.exists(path):
    if path != default_path and os.path.exists(default_path):
        print(f"Chemin introuvable. Utilisation du fichier par défaut : {default_path}")
        path = default_path
    else:
        raise FileNotFoundError(f"Le fichier {os.path.basename(default_path)} n'existe pas dans le dossier {default_dir}")

In [328]:
dataframe = pd.read_csv(path)

In [329]:
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Poster_Link    1000 non-null   object 
 1   Series_Title   1000 non-null   object 
 2   Released_Year  1000 non-null   object 
 3   Certificate    899 non-null    object 
 4   Runtime        1000 non-null   object 
 5   Genre          1000 non-null   object 
 6   IMDB_Rating    1000 non-null   float64
 7   Overview       1000 non-null   object 
 8   Meta_score     843 non-null    float64
 9   Director       1000 non-null   object 
 10  Star1          1000 non-null   object 
 11  Star2          1000 non-null   object 
 12  Star3          1000 non-null   object 
 13  Star4          1000 non-null   object 
 14  No_of_Votes    1000 non-null   int64  
 15  Gross          831 non-null    object 
dtypes: float64(2), int64(1), object(13)
memory usage: 125.1+ KB


In [330]:
dataframe.columns

Index(['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate',
       'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director',
       'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross'],
      dtype='object')

In [331]:
dataframe.head(3)

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"


### Études des données
- choix des features pertinentes pour la modélisation
    + variables pertinentes à la classification
        - année de sortie
        - certification (UA, A, U, etc.)
        - durée
        - IMDB note
        - Meta score
        - réalisteur (à transformer en valeurs numériques → realisateur ID)
        - nombre de votes
        - revenu brut (gross income)
    + variable cible
        - genre (après uniformisation/encodage)

In [332]:
sub_dataframe = dataframe[['Released_Year', 'Director', 'Certificate', 'Runtime', 'IMDB_Rating', 'Meta_score', 'No_of_Votes', 'Gross', 'Genre']]

In [333]:
sub_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  1000 non-null   object 
 1   Director       1000 non-null   object 
 2   Certificate    899 non-null    object 
 3   Runtime        1000 non-null   object 
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     843 non-null    float64
 6   No_of_Votes    1000 non-null   int64  
 7   Gross          831 non-null    object 
 8   Genre          1000 non-null   object 
dtypes: float64(2), int64(1), object(6)
memory usage: 70.4+ KB


- variables pertinentes à la classification

In [334]:
sub_dataframe[['Released_Year', 'Director', 'Certificate', 'Runtime', 'IMDB_Rating', 'Meta_score', 'No_of_Votes', 'Gross']].head(3)

,Released_Year,Director,Certificate,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross
0,1994,Frank Darabont,A,142 min,9.3,80.0,2343110,"28,341,469"
1,1972,Francis Ford Coppola,A,175 min,9.2,100.0,1620367,"134,966,411"
2,2008,Christopher Nolan,UA,152 min,9.0,84.0,2303232,"534,858,444"


- variable cible 

In [335]:
sub_dataframe['Genre'].head(5)

0                   Drama
1            Crime, Drama
2    Action, Crime, Drama
3            Crime, Drama
4            Crime, Drama
Name: Genre, dtype: object

### Traitement des données

In [336]:
# sub_dataframe = sub_dataframe.drop_duplicates()

- valeurs nulles par colonnes

In [337]:
print("Nombre de valeurs nulles par colonne:")
print(sub_dataframe[['Released_Year', 'Director', 'Certificate', 'Runtime', 'IMDB_Rating', 'Meta_score', 'No_of_Votes', 'Gross']].isnull().sum())

Nombre de valeurs nulles par colonne:
Released_Year      0
Director           0
Certificate      101
Runtime            0
IMDB_Rating        0
Meta_score       157
No_of_Votes        0
Gross            169
dtype: int64


#### uniformisation des durées de film

In [338]:
valeur_uniques_duration = sub_dataframe['Runtime'].unique()
valeur_uniques_duration

array(['142 min', '175 min', '152 min', '202 min', '96 min', '201 min',
       '154 min', '195 min', '148 min', '139 min', '178 min', '161 min',
       '179 min', '136 min', '146 min', '124 min', '133 min', '160 min',
       '132 min', '153 min', '169 min', '130 min', '125 min', '189 min',
       '116 min', '127 min', '118 min', '121 min', '207 min', '122 min',
       '106 min', '112 min', '151 min', '150 min', '155 min', '119 min',
       '110 min', '88 min', '137 min', '89 min', '165 min', '109 min',
       '102 min', '87 min', '126 min', '147 min', '117 min', '181 min',
       '149 min', '105 min', '164 min', '170 min', '98 min', '101 min',
       '113 min', '134 min', '229 min', '115 min', '143 min', '95 min',
       '104 min', '123 min', '131 min', '108 min', '81 min', '99 min',
       '114 min', '129 min', '228 min', '128 min', '103 min', '107 min',
       '68 min', '138 min', '156 min', '167 min', '163 min', '186 min',
       '321 min', '135 min', '140 min', '180 min', '158 min'

In [339]:
def convertir_duree(duree_str):
    if pd.isna(duree_str) or not isinstance(duree_str, str):
        return float('nan')

    duree_str = duree_str.strip().lower().replace(",", "")

    if "h" in duree_str:
        parts = duree_str.split("h", 1)
        heures = int(parts[0].strip()) if parts[0].strip() else 0
        minutes_part = parts[1].replace("min", "").strip() if len(parts) > 1 else ""
        minutes = int(minutes_part) if minutes_part else 0
        return heures * 60 + minutes

    if "min" in duree_str:
        minutes_str = duree_str.replace("min", "").strip()
        return int(minutes_str) if minutes_str else 0

    if duree_str.isdigit():
        return int(duree_str)

    return float('nan')

# test
# for duree in valeur_uniques_duration:
#     print(f"{duree} -> {convertir_duree(duree)}")

In [340]:
sub_dataframe.loc[:, 'Runtime'] = sub_dataframe['Runtime'].apply(convertir_duree)

In [341]:
# sub_dataframe.loc[:, 'duration'] = sub_dataframe['duration'].apply(lambda x: x if x > 0 else float('nan')) # remplacer les 0 par NaN

In [342]:
sub_dataframe.head(10)

,Released_Year,Director,Certificate,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,Genre
0,1994,Frank Darabont,A,142,9.3,80.0,2343110,"28,341,469",Drama
1,1972,Francis Ford Coppola,A,175,9.2,100.0,1620367,"134,966,411","Crime, Drama"
2,2008,Christopher Nolan,UA,152,9.0,84.0,2303232,"534,858,444","Action, Crime, Drama"
3,1974,Francis Ford Coppola,A,202,9.0,90.0,1129952,"57,300,000","Crime, Drama"
4,1957,Sidney Lumet,U,96,9.0,96.0,689845,"4,360,000","Crime, Drama"
5,2003,Peter Jackson,U,201,8.9,94.0,1642758,"377,845,905","Action, Adventure, Drama"
6,1994,Quentin Tarantino,A,154,8.9,94.0,1826188,"107,928,762","Crime, Drama"
7,1993,Steven Spielberg,A,195,8.9,94.0,1213505,"96,898,818","Biography, Drama, History"
8,2010,Christopher Nolan,UA,148,8.8,74.0,2067042,"292,576,195","Action, Adventure, Sci-Fi"
9,1999,David Fincher,A,139,8.8,66.0,1854740,"37,030,102",Drama


In [343]:
# appliquer le dtype int à la colonne 'Runtime' après conversion
sub_dataframe["Runtime"] = sub_dataframe["Runtime"].astype('float')
sub_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  1000 non-null   object 
 1   Director       1000 non-null   object 
 2   Certificate    899 non-null    object 
 3   Runtime        1000 non-null   float64
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     843 non-null    float64
 6   No_of_Votes    1000 non-null   int64  
 7   Gross          831 non-null    object 
 8   Genre          1000 non-null   object 
dtypes: float64(3), int64(1), object(5)
memory usage: 70.4+ KB


C:\Users\chari\AppData\Local\Temp\ipykernel_44524\4062814231.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_dataframe["Runtime"] = sub_dataframe["Runtime"].astype('float')


In [344]:
# outliers = get_outlier_by_iqr(sub_dataframe, 'Runtime')
# print("Outliers détectés dans la colonne 'Runtime':")
# print(outliers[['Runtime']])

#### uniformisation des années de sortie

In [345]:
def convertir_annee(annee_str):
    if pd.isna(annee_str):
        return float('nan')

    # gérer aussi les cas numériques déjà propres
    if isinstance(annee_str, (int, float)) and not pd.isna(annee_str):
        annee_int = int(annee_str)
        return annee_int if 1800 <= annee_int <= 2100 else float('nan')

    if not isinstance(annee_str, str):
        return float('nan')

    texte = annee_str.strip()

    # récupérer la première année à 4 chiffres trouvée dans la chaîne
    match = re.search(r'(?<!\d)(\d{4})(?!\d)', texte)
    if match:
        annee_int = int(match.group(1))
        return annee_int if 1800 <= annee_int <= 2100 else float('nan')

    return float('nan')

# test (aperçu limité pour éviter un affichage trop long)
# valeur_uniques_year = sub_dataframe['year'].dropna().unique()
# for annee in valeur_uniques_year[:50]:
#     print(f"{annee} -> {convertir_annee(annee)}")

# print("Valeurs 'year' non converties (NaN) :", sub_dataframe['year'].isna().sum())

In [346]:
sub_dataframe.loc[:, 'Released_Year'] = sub_dataframe['Released_Year'].apply(convertir_annee)

In [347]:
sub_dataframe.head(10)

,Released_Year,Director,Certificate,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,Genre
0,1994.0,Frank Darabont,A,142.0,9.3,80.0,2343110,"28,341,469",Drama
1,1972.0,Francis Ford Coppola,A,175.0,9.2,100.0,1620367,"134,966,411","Crime, Drama"
2,2008.0,Christopher Nolan,UA,152.0,9.0,84.0,2303232,"534,858,444","Action, Crime, Drama"
3,1974.0,Francis Ford Coppola,A,202.0,9.0,90.0,1129952,"57,300,000","Crime, Drama"
4,1957.0,Sidney Lumet,U,96.0,9.0,96.0,689845,"4,360,000","Crime, Drama"
5,2003.0,Peter Jackson,U,201.0,8.9,94.0,1642758,"377,845,905","Action, Adventure, Drama"
6,1994.0,Quentin Tarantino,A,154.0,8.9,94.0,1826188,"107,928,762","Crime, Drama"
7,1993.0,Steven Spielberg,A,195.0,8.9,94.0,1213505,"96,898,818","Biography, Drama, History"
8,2010.0,Christopher Nolan,UA,148.0,8.8,74.0,2067042,"292,576,195","Action, Adventure, Sci-Fi"
9,1999.0,David Fincher,A,139.0,8.8,66.0,1854740,"37,030,102",Drama


In [348]:
sub_dataframe["Released_Year"] = sub_dataframe["Released_Year"].astype('float')
sub_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  999 non-null    float64
 1   Director       1000 non-null   object 
 2   Certificate    899 non-null    object 
 3   Runtime        1000 non-null   float64
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     843 non-null    float64
 6   No_of_Votes    1000 non-null   int64  
 7   Gross          831 non-null    object 
 8   Genre          1000 non-null   object 
dtypes: float64(4), int64(1), object(4)
memory usage: 70.4+ KB


C:\Users\chari\AppData\Local\Temp\ipykernel_44524\1481997847.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_dataframe["Released_Year"] = sub_dataframe["Released_Year"].astype('float')


#### uniformisation des votes

In [349]:
sub_dataframe["No_of_Votes"].unique()

array([2343110, 1620367, 2303232, 1129952,  689845, 1642758, 1826188,
       1213505, 2067042, 1854740, 1661481, 1809221,  688390, 1485555,
       1676426, 1020727, 1159315,  918088,   55291,  552778,   54995,
       1512360,  699256,  651376, 1235804, 1147794,  623629, 1445096,
       1270197, 1231473,   42004,  315744,  405801,  939252,  717585,
        760360, 1190259, 1189773,  729603, 1341460, 1034705,  991208,
       1035236,  942045,  995506,  230763,  235231, 1058081,  302844,
        604211,  522093,  217881,  167839,   62635,   34112,   28401,
        194838,  156479,  375110,  809955,  834477,  384171, 1357682,
       1516346,  344445,  168895,  999790,  358685,  515451, 1125712,
        343171,  311365,  884112,  898237,  606398,  787806,   30273,
         34357,  450474,  108862,  178092,  444074,  201632,  203150,
        425844,   27793,   71875,   30722,  281623,  220002,  150023,
         33935,   78925, 1267869,  911664,  703810,  782001,  766870,
       1069738,  861

In [350]:
def convertir_votes(votes_str):
    if pd.isna(votes_str):
        return float('nan')

    # cas numérique direct (int/float)
    if isinstance(votes_str, (int, float)):
        votes_float = float(votes_str)
        if votes_float < 0 or not votes_float.is_integer():
            return float('nan')
        return int(votes_float)

    # cas non string non numérique
    if not isinstance(votes_str, str):
        return float('nan')

    # nettoyage texte
    texte = votes_str.strip().replace(",", "")
    if not texte:
        return float('nan')

    # conversion robuste (gère "124.0", "22792", etc.)
    try:
        votes_float = float(texte)
    except ValueError:
        return float('nan')

    if votes_float < 0 or not votes_float.is_integer():
        return float('nan')

    return int(votes_float)

# test
# valeur_uniques_votes = sub_dataframe['votes'].dropna().unique()
# for votes in valeur_uniques_votes[:50]:
#     print(f"{votes} -> {convertir_votes(votes)}")

In [351]:
sub_dataframe.loc[:, 'No_of_Votes'] = sub_dataframe['No_of_Votes'].apply(convertir_votes)

In [352]:
sub_dataframe.head(10)

,Released_Year,Director,Certificate,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,Genre
0,1994.0,Frank Darabont,A,142.0,9.3,80.0,2343110,"28,341,469",Drama
1,1972.0,Francis Ford Coppola,A,175.0,9.2,100.0,1620367,"134,966,411","Crime, Drama"
2,2008.0,Christopher Nolan,UA,152.0,9.0,84.0,2303232,"534,858,444","Action, Crime, Drama"
3,1974.0,Francis Ford Coppola,A,202.0,9.0,90.0,1129952,"57,300,000","Crime, Drama"
4,1957.0,Sidney Lumet,U,96.0,9.0,96.0,689845,"4,360,000","Crime, Drama"
5,2003.0,Peter Jackson,U,201.0,8.9,94.0,1642758,"377,845,905","Action, Adventure, Drama"
6,1994.0,Quentin Tarantino,A,154.0,8.9,94.0,1826188,"107,928,762","Crime, Drama"
7,1993.0,Steven Spielberg,A,195.0,8.9,94.0,1213505,"96,898,818","Biography, Drama, History"
8,2010.0,Christopher Nolan,UA,148.0,8.8,74.0,2067042,"292,576,195","Action, Adventure, Sci-Fi"
9,1999.0,David Fincher,A,139.0,8.8,66.0,1854740,"37,030,102",Drama


In [353]:
sub_dataframe["No_of_Votes"] = sub_dataframe["No_of_Votes"].astype('float')
sub_dataframe.info()

C:\Users\chari\AppData\Local\Temp\ipykernel_44524\155778270.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_dataframe["No_of_Votes"] = sub_dataframe["No_of_Votes"].astype('float')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  999 non-null    float64
 1   Director       1000 non-null   object 
 2   Certificate    899 non-null    object 
 3   Runtime        1000 non-null   float64
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     843 non-null    float64
 6   No_of_Votes    1000 non-null   float64
 7   Gross          831 non-null    object 
 8   Genre          1000 non-null   object 
dtypes: float64(5), object(4)
memory usage: 70.4+ KB


In [354]:
outliers = get_outlier_by_iqr(sub_dataframe, 'No_of_Votes')
print("Outliers détectés dans la colonne 'No_of_Votes':")
print(outliers[['No_of_Votes']], "\n", "Nombre d'outliers:", len(outliers))

Outliers détectés dans la colonne 'No_of_Votes':
     No_of_Votes
0      2343110.0
1      1620367.0
2      2303232.0
3      1129952.0
5      1642758.0
..           ...
376    1015122.0
477     860823.0
502     939644.0
623    1118998.0
652    1046089.0

[67 rows x 1 columns] 
 Nombre d'outliers: 67


In [355]:
sub_dataframe.loc[:, 'No_of_Votes'] = sub_dataframe['No_of_Votes'].apply(
                                                            # utilier log(x + 1) pour éviter les problèmes de log(0) et log(negative)
                                                            lambda x: np.log(x + 1)
                                                    )
sub_dataframe.head(10)

,Released_Year,Director,Certificate,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,Genre
0,1994.0,Frank Darabont,A,142.0,9.3,80.0,14.666990,"28,341,469",Drama
1,1972.0,Francis Ford Coppola,A,175.0,9.2,100.0,14.298164,"134,966,411","Crime, Drama"
2,2008.0,Christopher Nolan,UA,152.0,9.0,84.0,14.649824,"534,858,444","Action, Crime, Drama"
3,1974.0,Francis Ford Coppola,A,202.0,9.0,90.0,13.937687,"57,300,000","Crime, Drama"
4,1957.0,Sidney Lumet,U,96.0,9.0,96.0,13.444224,"4,360,000","Crime, Drama"
5,2003.0,Peter Jackson,U,201.0,8.9,94.0,14.311888,"377,845,905","Action, Adventure, Drama"
6,1994.0,Quentin Tarantino,A,154.0,8.9,94.0,14.417742,"107,928,762","Crime, Drama"
7,1993.0,Steven Spielberg,A,195.0,8.9,94.0,14.009024,"96,898,818","Biography, Drama, History"
8,2010.0,Christopher Nolan,UA,148.0,8.8,74.0,14.541630,"292,576,195","Action, Adventure, Sci-Fi"
9,1999.0,David Fincher,A,139.0,8.8,66.0,14.433256,"37,030,102",Drama


#### uniformisation des revenus bruts

In [356]:
def convertir_gross_income(gross_str):
    if pd.isna(gross_str):
        return float('nan')

    # déjà numérique
    if isinstance(gross_str, (int, float)):
        return float(gross_str)

    if not isinstance(gross_str, str):
        return float('nan')

    s = gross_str.strip()
    if not s:
        return float('nan')

    # enlever séparateurs de milliers et caractères non numériques
    s = s.replace(",", "")
    s = re.sub(r"[^\d.\-]", "", s)

    if s in {"", ".", "-", "-."}:
        return float('nan')

    try:
        return float(s)
    
    except ValueError:
        return float('nan')

In [357]:
sub_dataframe["Gross"].unique()

array(['28,341,469', '134,966,411', '534,858,444', '57,300,000',
       '4,360,000', '377,845,905', '107,928,762', '96,898,818',
       '292,576,195', '37,030,102', '315,544,750', '330,252,182',
       '6,100,000', '342,551,365', '171,479,930', '46,836,394',
       '290,475,067', '112,000,000', nan, '53,367,844', '188,020,017',
       '7,563,397', '10,055,859', '216,540,909', '136,801,374',
       '57,598,247', '100,125,643', '130,742,922', '322,740,140',
       '269,061', '335,451,311', '13,092,000', '13,182,281', '53,089,891',
       '132,384,315', '32,572,577', '187,705,427', '6,719,864',
       '23,341,568', '19,501,238', '422,783,777', '204,843,350',
       '11,990,401', '210,609,762', '5,321,508', '32,000,000',
       '1,024,560', '163,245', '19,181', '1,661,096', '5,017,246',
       '12,391,761', '190,241,310', '858,373,000', '678,815,482',
       '209,726,015', '162,805,434', '448,139,099', '6,532,908',
       '1,223,869', '223,808,164', '11,286,112', '707,481', '25,544,867',
 

In [358]:
sub_dataframe.loc[:, 'Gross'] = sub_dataframe['Gross'].apply(convertir_gross_income)

In [359]:
sub_dataframe["Gross"].unique().sort()

In [360]:
# combien de vide/NaN dans cette colonne ?
print(sub_dataframe['Gross'].isnull().sum())

169


In [361]:
sub_dataframe.loc[:, 'Gross'] = sub_dataframe['Gross'].apply(lambda x: x if x >= 0 else float('nan')) # remplacer les 0 par NaN

In [362]:
# combien de vide/NaN dans cette colonne ?
# print(sub_dataframe['Gross'].isnull().sum())

In [363]:
sub_dataframe.head(10)

,Released_Year,Director,Certificate,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,Genre
0,1994.0,Frank Darabont,A,142.0,9.3,80.0,14.666990,28341469.0,Drama
1,1972.0,Francis Ford Coppola,A,175.0,9.2,100.0,14.298164,134966411.0,"Crime, Drama"
2,2008.0,Christopher Nolan,UA,152.0,9.0,84.0,14.649824,534858444.0,"Action, Crime, Drama"
3,1974.0,Francis Ford Coppola,A,202.0,9.0,90.0,13.937687,57300000.0,"Crime, Drama"
4,1957.0,Sidney Lumet,U,96.0,9.0,96.0,13.444224,4360000.0,"Crime, Drama"
5,2003.0,Peter Jackson,U,201.0,8.9,94.0,14.311888,377845905.0,"Action, Adventure, Drama"
6,1994.0,Quentin Tarantino,A,154.0,8.9,94.0,14.417742,107928762.0,"Crime, Drama"
7,1993.0,Steven Spielberg,A,195.0,8.9,94.0,14.009024,96898818.0,"Biography, Drama, History"
8,2010.0,Christopher Nolan,UA,148.0,8.8,74.0,14.541630,292576195.0,"Action, Adventure, Sci-Fi"
9,1999.0,David Fincher,A,139.0,8.8,66.0,14.433256,37030102.0,Drama


In [364]:
sub_dataframe["Gross"] = sub_dataframe["Gross"].astype('float')
sub_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  999 non-null    float64
 1   Director       1000 non-null   object 
 2   Certificate    899 non-null    object 
 3   Runtime        1000 non-null   float64
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     843 non-null    float64
 6   No_of_Votes    1000 non-null   float64
 7   Gross          831 non-null    float64
 8   Genre          1000 non-null   object 
dtypes: float64(6), object(3)
memory usage: 70.4+ KB


C:\Users\chari\AppData\Local\Temp\ipykernel_44524\1458042169.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_dataframe["Gross"] = sub_dataframe["Gross"].astype('float')


In [365]:
outliers = get_outlier_by_iqr(sub_dataframe, 'Gross')
print("Outliers détectés dans la colonne 'Gross':")
print(outliers[['Gross']], "\n", "Nombre d'outliers:", len(outliers))

Outliers détectés dans la colonne 'Gross':
           Gross
2    534858444.0
5    377845905.0
8    292576195.0
10   315544750.0
11   330252182.0
..           ...
915  255959475.0
927  301959197.0
928  210614939.0
947  317575550.0
973  285761243.0

[89 rows x 1 columns] 
 Nombre d'outliers: 89


In [366]:
sub_dataframe.loc[:, 'Gross'] = sub_dataframe['Gross'].apply(
                                    # utilier log(x + 1) pour éviter les problèmes de log(0) et log(negative)
                                    lambda x: np.log(x + 1)
                                    )
sub_dataframe.head(10)

,Released_Year,Director,Certificate,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,Genre
0,1994.0,Frank Darabont,A,142.0,9.3,80.0,14.666990,17.159837,Drama
1,1972.0,Francis Ford Coppola,A,175.0,9.2,100.0,14.298164,18.720537,"Crime, Drama"
2,2008.0,Christopher Nolan,UA,152.0,9.0,84.0,14.649824,20.097513,"Action, Crime, Drama"
3,1974.0,Francis Ford Coppola,A,202.0,9.0,90.0,13.937687,17.863811,"Crime, Drama"
4,1957.0,Sidney Lumet,U,96.0,9.0,96.0,13.444224,15.287983,"Crime, Drama"
5,2003.0,Peter Jackson,U,201.0,8.9,94.0,14.311888,19.749997,"Action, Adventure, Drama"
6,1994.0,Quentin Tarantino,A,154.0,8.9,94.0,14.417742,18.496982,"Crime, Drama"
7,1993.0,Steven Spielberg,A,195.0,8.9,94.0,14.009024,18.389178,"Biography, Drama, History"
8,2010.0,Christopher Nolan,UA,148.0,8.8,74.0,14.541630,19.494236,"Action, Adventure, Sci-Fi"
9,1999.0,David Fincher,A,139.0,8.8,66.0,14.433256,17.427242,Drama


#### uniformisation des directeurs

In [367]:
sub_dataframe['Director'].unique()

array(['Frank Darabont', 'Francis Ford Coppola', 'Christopher Nolan',
       'Sidney Lumet', 'Peter Jackson', 'Quentin Tarantino',
       'Steven Spielberg', 'David Fincher', 'Robert Zemeckis',
       'Sergio Leone', 'Lana Wachowski', 'Martin Scorsese',
       'Irvin Kershner', 'Milos Forman', 'Thomas Kail', 'Bong Joon Ho',
       'Sudha Kongara', 'Fernando Meirelles', 'Hayao Miyazaki',
       'Roberto Benigni', 'Jonathan Demme', 'George Lucas',
       'Masaki Kobayashi', 'Akira Kurosawa', 'Frank Capra',
       'Todd Phillips', 'Damien Chazelle', 'Olivier Nakache',
       'Roman Polanski', 'Ridley Scott', 'Tony Kaye', 'Bryan Singer',
       'Luc Besson', 'Roger Allers', 'James Cameron',
       'Giuseppe Tornatore', 'Isao Takahata', 'Alfred Hitchcock',
       'Michael Curtiz', 'Charles Chaplin', 'Nadine Labaki', 'Can Ulkay',
       'Gayatri', 'Makoto Shinkai', 'Nitesh Tiwari', 'Bob Persichetti',
       'Anthony Russo', 'Lee Unkrich', 'Rajkumar Hirani', 'Aamir Khan',
       'Andrew Stant

In [368]:
# def fix_mojibake_text(x):
#     if not isinstance(x, str):
#         return x

#     s = x.strip()
#     if not s:
#         return s

#     # détecter des marqueurs fréquents de texte mal décodé (ex: "GarcÃ­a", "JosÃ©")
#     markers = ("Ã", "Â", "â", "¤", "�")
#     if any(m in s for m in markers):
#         for source_encoding in ("latin1", "cp1252"):
#             try:
#                 repaired = s.encode(source_encoding).decode("utf-8")
#                 before = sum(s.count(m) for m in markers)
#                 after = sum(repaired.count(m) for m in markers)
#                 if after < before:
#                     s = repaired
#                     break
#             except (UnicodeEncodeError, UnicodeDecodeError):
#                 continue

#     return s

In [369]:
# _MOJIBAKE_RE = re.compile(
#     r"[\xc0-\xc3\xc5\xc6\xc8-\xcf\xd0-\xd6\xd8-\xdd\xe0-\xef\xf0-\xf6\xf8-\xfd]"
#     r"[\x80-\xbf]",
#     re.UNICODE,
# )

# _OBVIOUS_MARKERS = frozenset([
#     "Ã", "Â", "â€", "Ã©", "Ã¨", "Ã ", "Ã¢", "Ã®", "Ã´", "Ã»",
#     "Ã‡", "Ã±", "Ã¼", "Ã¶", "Ã„", "Ã–", "Ãœ",
#     "â€™", "â€œ", "â€\x9d", "â€“", "â€”",
#     "Â«", "Â»", "Â°", "Â·",
#     "ï»¿", "â", "¤", "¿", "½", "¼", "¾",
#     "\ufffd",  # Replacement character (U+FFFD)
#     "\xc3\xbb",      # û (Ã»)
#     "\xc3\x87",      # Ç (Ã‡)
#     "\xc3\xb1",      # ñ (Ã±)
#     "\xc3\xbc",      # ü (Ã¼)
#     "\xc3\xb6",      # ö (Ã¶)
#     "\xc3\x84",      # Ä (Ã„)
#     "\xc3\x96",      # Ö (Ã–)
#     "\xc3\x9c",      # Ü (Ãœ)
#     # Guillemets et tirets mal décodés
#     "\xe2\x80\x99",  # ' (â€™, apostrophe)
#     "\xe2\x80\x9c",  # " (â€œ, guillemet ouvrant)
#     "\xe2\x80\x9d",  # " (â€, guillemet fermant)
#     "\xe2\x80\x93",  # – (â€", tiret court)
#     "\xe2\x80\x94",  # — (â€", tiret long)
#     # Autres caractères spéciaux
#     "\xc2\xab",      # « (Â«)
#     "\xc2\xbb",      # » (Â»)
#     "\xc2\xb0",      # ° (Â°)
#     "\xc2\xb7",      # · (Â·)
#     "\xef\xbb\xbf",  # BOM UTF-8 (ï»¿)
#     "\xe2\x80",      # Début de séquence mal décodée (â€)
#     "\xc2",          # Début de séquence mal décodée (Â)
#     "\ufffd",        # Replacement character (U+FFFD)
# ])

# _SOURCE_ENCODINGS = ["utf-8", "latin1", "cp1252", "cp850", "iso-8859-15", "mac_roman"]
# _TARGET_ENCODING = "utf-8"

# MAX_PASSES = 3

# def _has_mojibake(s: str) -> bool:
#     """Détecte si la chaîne contient probablement du mojibake."""
#     if any(marker in s for marker in _OBVIOUS_MARKERS):
#         return True
#     if _MOJIBAKE_RE.search(s):
#         return True
#     return False


# def _unicode_score(s: str) -> float:
#     """
#     Score de qualité Unicode : ratio de caractères 'normaux'
#     (lettres, chiffres, ponctuation courante) vs caractères suspects.
#     Plus le score est élevé, meilleur est le texte.
#     """
#     if not s:
#         return 1.0
#     good = sum(
#         1 for c in s
#         if unicodedata.category(c) in {
#             "Ll", "Lu", "Lt", "Lm", "Lo",  # Lettres
#             "Nd", "Nl", "No",              # Chiffres
#             "Pc", "Pd", "Pe", "Pf", "Pi", "Po", "Ps",  # Ponctuation
#             "Zs",                          # Espaces
#             "Sm", "Sc", "So",             # Symboles courants
#         }
#     )
#     return good / len(s)


# def _try_repair(s: str) -> str:
#     """Tente une passe de réparation en testant plusieurs encodages source."""
#     best = s
#     best_score = _unicode_score(s)

#     for source_enc in _SOURCE_ENCODINGS:
#         try:
#             candidate = s.encode(source_enc).decode(_TARGET_ENCODING)
#         except (UnicodeEncodeError, UnicodeDecodeError):
#             continue

#         # Ignorer si le résultat est identique ou plus long de façon suspecte
#         if candidate == s:
#             continue

#         score = _unicode_score(candidate)
#         if score > best_score + 0.05:  # Seuil minimal d'amélioration
#             best = candidate
#             best_score = score

#     return best


# def fix_mojibake_text(x):
#     """
#     Répare le mojibake dans une chaîne de caractères.

#     Gère :
#     - Encodages simples et doubles (multi-passes)
#     - latin1, cp1252, cp850, iso-8859-15, mac_roman → UTF-8
#     - Entités HTML (&eacute;, &#233;, &#x00E9;)
#     - BOM UTF-8 (\\ufeff)
#     - Normalisation Unicode NFC
#     """
#     if not isinstance(x, str):
#         return x

#     s = x.strip()
#     if not s:
#         return s

#     # 1. Supprimer le BOM UTF-8 si présent
#     s = s.lstrip("\ufeff")

#     # 2. Décoder les entités HTML (ex: &eacute; → é, &#233; → é)
#     s_html = html.unescape(s)
#     if s_html != s and _unicode_score(s_html) >= _unicode_score(s):
#         s = s_html

#     # 3. Réparation itérative du mojibake (multi-passes pour double encodage)
#     for _ in range(MAX_PASSES):
#         if not _has_mojibake(s):
#             break
#         repaired = _try_repair(s)
#         if repaired == s:
#             break
#         s = repaired

#     # 4. Normalisation Unicode NFC (compose les caractères décomposés)
#     s = unicodedata.normalize("NFC", s)

#     return s

In [370]:
_MOJIBAKE_RE = re.compile(
    r"[\xc0-\xc3\xc5\xc6\xc8-\xcf\xd0-\xd6\xd8-\xdd\xe0-\xef\xf0-\xf6\xf8-\xfd]"
    r"[\x80-\xbf]",
    re.UNICODE,
)

_OBVIOUS_MARKERS = frozenset([
    "Ã", "Â", "â€", "Ã©", "Ã¨", "Ã ", "Ã¢", "Ã®", "Ã´", "Ã»",
    "Ã‡", "Ã±", "Ã¼", "Ã¶", "Ã„", "Ã–", "Ãœ",
    "â€™", "â€œ", "â€\x9c", "â€\x9d",
    "Â«", "Â»", "Â°", "Â·",
    "ï»¿", "â", "¤", "¿", "½", "¼", "¾",
    "\ufffd"])

_SOURCE_ENCODINGS = ["latin1", "cp1252", "cp850", "iso-8859-15", "mac_roman"]
MAX_PASSES = 3

def _has_mojibake(s: str) -> bool:
    if any(marker in s for marker in _OBVIOUS_MARKERS):
        return True
    return bool(_MOJIBAKE_RE.search(s))


def _unicode_score(s: str) -> float:
    if not s:
        return 1.0
    good = sum(
        1 for c in s
        if unicodedata.category(c) in {
            "Ll", "Lu", "Lt", "Lm", "Lo",
            "Nd", "Nl", "No",
            "Pc", "Pd", "Pe", "Pf", "Pi", "Po", "Ps",
            "Zs", "Sm", "Sc", "So",
        }
    )
    return good / len(s)


def _try_repair(s: str) -> str | None:
    """
    Retourne la meilleure réparation trouvée, ou None si aucune n'est meilleure.
    Logique assouplie : on accepte dès que le score ne régresse pas
    ET qu'il n'y a pas de caractère de remplacement U+FFFD introduit.
    """
    best = None
    best_score = _unicode_score(s)

    for source_enc in _SOURCE_ENCODINGS:
        try:
            candidate = s.encode(source_enc).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue

        if candidate == s:
            continue

        # Rejeter si la réparation introduit des caractères de remplacement
        if "\ufffd" in candidate:
            continue

        score = _unicode_score(candidate)

        # ✅ Seuil assoupli : accepter si score ≥ original (sans marge arbitraire)
        # car Ã/©/¡ sont valides → les deux côtés ont un score similaire
        if score >= best_score:
            best = candidate
            best_score = score

    return best


def fix_mojibake_text(x):
    if not isinstance(x, str):
        return x

    s = x.strip()
    if not s:
        return s

    # 1. Supprimer le BOM UTF-8
    s = s.lstrip("\ufeff")

    # 2. Décoder les entités HTML (&eacute; → é)
    s_html = html.unescape(s)
    if s_html != s and "\ufffd" not in s_html:
        s = s_html

    # 3. Réparation itérative (multi-passes pour double mojibake)
    for _ in range(MAX_PASSES):
        if not _has_mojibake(s):
            break
        repaired = _try_repair(s)
        if repaired is None:
            break
        s = repaired

    # 4. Normalisation NFC
    s = unicodedata.normalize("NFC", s)

    return s

In [371]:
sub_dataframe.loc[:, 'Director'] = sub_dataframe['Director'].apply(
    # gérer les unicodes et les caractères spéciaux dans les noms et prénoms de réalisateurs
    fix_mojibake_text
)

In [372]:
sub_dataframe = uniformisation_string_to_int_encoding(sub_dataframe, 'Director', "DirectorID")

C:\Users\chari\AppData\Local\Temp\ipykernel_44524\3839918192.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe.loc[:, new_column_name] = label_encoder.fit_transform(dataframe[column].astype(str))


In [373]:
# transformer les noms de directeurs en minuscules et enlever les espaces superflus
# sub_dataframe.loc[:, 'Director'] = sub_dataframe['Director'].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)
# sub_dataframe['Director'].unique()

In [374]:
# transformer les directeurs en catégories numériques
# sub_dataframe.loc[:, 'Director_ID'] = label_encoder.fit_transform(sub_dataframe['Director'].astype(str))
# sub_dataframe.drop(columns=['Director'], inplace=True)
sub_dataframe.head(10)

,Released_Year,Director,Certificate,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,Genre,DirectorID
0,1994.0,frank darabont,A,142.0,9.3,80.0,14.666990,17.159837,Drama,141
1,1972.0,francis ford coppola,A,175.0,9.2,100.0,14.298164,18.720537,"Crime, Drama",137
2,2008.0,christopher nolan,UA,152.0,9.0,84.0,14.649824,20.097513,"Action, Crime, Drama",83
3,1974.0,francis ford coppola,A,202.0,9.0,90.0,13.937687,17.863811,"Crime, Drama",137
4,1957.0,sidney lumet,U,96.0,9.0,96.0,13.444224,15.287983,"Crime, Drama",456
5,2003.0,peter jackson,U,201.0,8.9,94.0,14.311888,19.749997,"Action, Adventure, Drama",383
6,1994.0,quentin tarantino,A,154.0,8.9,94.0,14.417742,18.496982,"Crime, Drama",391
7,1993.0,steven spielberg,A,195.0,8.9,94.0,14.009024,18.389178,"Biography, Drama, History",470
8,2010.0,christopher nolan,UA,148.0,8.8,74.0,14.541630,19.494236,"Action, Adventure, Sci-Fi",83
9,1999.0,david fincher,A,139.0,8.8,66.0,14.433256,17.427242,Drama,100


In [375]:
sub_dataframe["DirectorID"] = sub_dataframe["DirectorID"].astype('float')

C:\Users\chari\AppData\Local\Temp\ipykernel_44524\3358962110.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_dataframe["DirectorID"] = sub_dataframe["DirectorID"].astype('float')


#### uniformisation des certifications

In [376]:
sub_dataframe["Certificate"].unique()

array(['A', 'UA', 'U', 'PG-13', 'R', nan, 'PG', 'G', 'Passed', 'TV-14',
       '16', 'TV-MA', 'Unrated', 'GP', 'Approved', 'TV-PG', 'U/A'],
      dtype=object)

- uniformer les certifications (UA, A, U, etc.) en un format universel

In [377]:
CERTIFICATION_MAP = {
    # Tout public
    "U":        "U",
    "G":        "U",
    "PASSED":   "U",       # Validé sans restriction explicite
    "APPROVED": "U",

    # Accord parental
    "UA":       "PG",
    "U/A":      "PG",
    "PG":       "PG",
    "GP":       "PG",       # Ancien code MPAA (avant PG)
    "TV-PG":    "PG",

    # 13 ans et plus
    "PG-13":    "13+",
    "TV-14":    "13+",

    # 16 ans et plus
    "16":       "16+",

    # 18 ans et plus
    "A":        "18+",
    "R":        "18+",
    "TV-MA":    "18+",

    # Non classifié
    "UNRATED":  "NR",
}

In [378]:
def uniformer_certification(cert_str):
    if pd.isna(cert_str):
        return CERTIFICATION_MAP.get("UNRATED")

    cert_str = cert_str.strip().upper()
    return CERTIFICATION_MAP.get(cert_str)

# test
valeur_uniques_cert = sub_dataframe['Certificate'].dropna().unique()
for cert in valeur_uniques_cert:
    print(f"{cert} -> {uniformer_certification(cert)}")

A -> 18+
UA -> PG
U -> U
PG-13 -> 13+
R -> 18+
PG -> PG
G -> U
Passed -> U
TV-14 -> 13+
16 -> 16+
TV-MA -> 18+
Unrated -> NR
GP -> PG
Approved -> U
TV-PG -> PG
U/A -> PG


In [379]:
sub_dataframe.loc[:, 'Certificate'] = sub_dataframe['Certificate'].apply(uniformer_certification)

In [380]:
# sub_dataframe.value_counts('Certificate', dropna=False)

In [381]:
sub_dataframe = uniformisation_string_to_int_encoding(sub_dataframe, 'Certificate', "CertificateID")

C:\Users\chari\AppData\Local\Temp\ipykernel_44524\3839918192.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe.loc[:, new_column_name] = label_encoder.fit_transform(dataframe[column].astype(str))


In [382]:
# sub_dataframe.drop(columns=['Certificate'], inplace=True)

In [383]:
sub_dataframe.head(10)

,Released_Year,Director,Certificate,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,Genre,DirectorID,CertificateID
0,1994.0,frank darabont,18+,142.0,9.3,80.0,14.666990,17.159837,Drama,141.0,2
1,1972.0,francis ford coppola,18+,175.0,9.2,100.0,14.298164,18.720537,"Crime, Drama",137.0,2
2,2008.0,christopher nolan,pg,152.0,9.0,84.0,14.649824,20.097513,"Action, Crime, Drama",83.0,4
3,1974.0,francis ford coppola,18+,202.0,9.0,90.0,13.937687,17.863811,"Crime, Drama",137.0,2
4,1957.0,sidney lumet,u,96.0,9.0,96.0,13.444224,15.287983,"Crime, Drama",456.0,5
5,2003.0,peter jackson,u,201.0,8.9,94.0,14.311888,19.749997,"Action, Adventure, Drama",383.0,5
6,1994.0,quentin tarantino,18+,154.0,8.9,94.0,14.417742,18.496982,"Crime, Drama",391.0,2
7,1993.0,steven spielberg,18+,195.0,8.9,94.0,14.009024,18.389178,"Biography, Drama, History",470.0,2
8,2010.0,christopher nolan,pg,148.0,8.8,74.0,14.541630,19.494236,"Action, Adventure, Sci-Fi",83.0,4
9,1999.0,david fincher,18+,139.0,8.8,66.0,14.433256,17.427242,Drama,100.0,2


In [384]:
sub_dataframe["CertificateID"] = sub_dataframe["CertificateID"].astype('float')

C:\Users\chari\AppData\Local\Temp\ipykernel_44524\1284431553.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub_dataframe["CertificateID"] = sub_dataframe["CertificateID"].astype('float')


In [385]:
sub_dataframe.info(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  999 non-null    float64
 1   Director       1000 non-null   object 
 2   Certificate    1000 non-null   object 
 3   Runtime        1000 non-null   float64
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     843 non-null    float64
 6   No_of_Votes    1000 non-null   float64
 7   Gross          831 non-null    float64
 8   Genre          1000 non-null   object 
 9   DirectorID     1000 non-null   float64
 10  CertificateID  1000 non-null   float64
dtypes: float64(8), object(3)
memory usage: 86.1+ KB


#### uniformisation des genres

In [386]:
print("Genres avant uniformisation :",
      sub_dataframe['Genre'].unique(),
      "\nNombre de genres uniques avant uniformisation :",
      len(sub_dataframe['Genre'].unique()))

Genres avant uniformisation : ['Drama' 'Crime, Drama' 'Action, Crime, Drama' 'Action, Adventure, Drama'
 'Biography, Drama, History' 'Action, Adventure, Sci-Fi' 'Drama, Romance'
 'Western' 'Action, Sci-Fi' 'Biography, Crime, Drama'
 'Action, Adventure, Fantasy' 'Comedy, Drama, Thriller'
 'Adventure, Drama, Sci-Fi' 'Animation, Adventure, Family' 'Drama, War'
 'Crime, Drama, Fantasy' 'Comedy, Drama, Romance' 'Crime, Drama, Mystery'
 'Crime, Drama, Thriller' 'Action, Drama, Mystery'
 'Drama, Family, Fantasy' 'Drama, Music' 'Biography, Comedy, Drama'
 'Drama, Mystery, Sci-Fi' 'Biography, Drama, Music'
 'Crime, Mystery, Thriller' 'Animation, Adventure, Drama'
 'Animation, Drama, War' 'Adventure, Comedy, Sci-Fi'
 'Horror, Mystery, Thriller' 'Drama, Romance, War' 'Comedy, Drama, Family'
 'Animation, Drama, Fantasy' 'Action, Biography, Drama'
 'Animation, Action, Adventure' 'Drama, Western' 'Action, Adventure'
 'Comedy, Drama' 'Drama, Family' 'Drama, Mystery, Thriller'
 'Mystery, Thriller' 'Dr

In [387]:
def extract_first_genre(x):
    if pd.isna(x) or not isinstance(x, str):
        return float('nan')
    return x.split(',')[0].strip()

# test
# for genre in sub_dataframe['Genre'].dropna().unique()[:50]:
#     print(f"{genre} -> {extract_first_genre(genre)}")

- filtrer les genres **multiples** selon une liste de genres indésirables
    1. prendre en entrée une chaîne de caractères représentant les genres d’un film, séparés par des virgules.
    2. conserver uniquement les genres qui ne figurent pas dans la liste des genres à supprimer (passée en paramètres).

In [388]:
liste_genres_a_retirer = {
    "Animation",
    "Reality-TV",
    "Talk-Show",
    "Game-Show",
    "Documentary",
    "News",
    "Film-Noir",
    "Adult",
    "Short",
    "Musical",
    "War",
    "Romance",
    "Crime",
    "Sci-Fi",
    "Family",
    "Fantasy",
    "Mystery",
    "Drama",
}

In [389]:
# fonction : tu prends en entrée une chaîne de caractères représentant les genres d'un film séparés par des virgules et tu vas garder uniquement les genres pas présents dans la liste en parametres
def filter_genres(genre_str, genres_to_remove):
    if pd.isna(genre_str) or not isinstance(genre_str, str):
        return float('nan')

    genres = [g.strip() for g in genre_str.split(",")]
    filtered_genres = [g for g in genres if g not in genres_to_remove]

    return ", ".join(filtered_genres) if filtered_genres else float('nan')

# test
# for genre_str in sub_dataframe['Genre'].unique():
#     print(f"{genre_str} -> {filter_genres(genre_str, liste_genres_a_retirer)}")

In [390]:
sub_dataframe['Genre'].value_counts()

Genre
Drama                        85
Drama, Romance               37
Comedy, Drama                35
Comedy, Drama, Romance       31
Action, Crime, Drama         30
                             ..
Action, Adventure, Family     1
Action, Crime, Mystery        1
Animation, Drama, Romance     1
Drama, War, Western           1
Adventure, Comedy, War        1
Name: count, Length: 202, dtype: int64

- garder uniquement le premier genre d’un film (après filtrage des genres indésirables)

In [396]:
sub_dataframe.loc[:, 'Genre'] = sub_dataframe['Genre'].apply(extract_first_genre)

In [397]:
sub_dataframe.value_counts('Genre')

Genre
Drama        289
Action       172
Comedy       155
Crime        107
Biography     88
Animation     82
Adventure     72
Mystery       12
Horror        11
Western        4
Film-Noir      3
Family         2
Fantasy        2
Thriller       1
Name: count, dtype: int64

In [398]:
sub_dataframe.head(10)

,Released_Year,Director,Certificate,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,Genre,DirectorID,CertificateID
0,1994.0,frank darabont,18+,142.0,9.3,80.0,14.666990,17.159837,Drama,141.0,2.0
1,1972.0,francis ford coppola,18+,175.0,9.2,100.0,14.298164,18.720537,Crime,137.0,2.0
2,2008.0,christopher nolan,pg,152.0,9.0,84.0,14.649824,20.097513,Action,83.0,4.0
3,1974.0,francis ford coppola,18+,202.0,9.0,90.0,13.937687,17.863811,Crime,137.0,2.0
4,1957.0,sidney lumet,u,96.0,9.0,96.0,13.444224,15.287983,Crime,456.0,5.0
5,2003.0,peter jackson,u,201.0,8.9,94.0,14.311888,19.749997,Action,383.0,5.0
6,1994.0,quentin tarantino,18+,154.0,8.9,94.0,14.417742,18.496982,Crime,391.0,2.0
7,1993.0,steven spielberg,18+,195.0,8.9,94.0,14.009024,18.389178,Biography,470.0,2.0
8,2010.0,christopher nolan,pg,148.0,8.8,74.0,14.541630,19.494236,Action,83.0,4.0
9,1999.0,david fincher,18+,139.0,8.8,66.0,14.433256,17.427242,Drama,100.0,2.0


In [ ]:
# retirer une liste de genres pour éviter d'avoir trop de classes différentes
genres_a_retirer = {
    "Animation",
    "Film-Noir",
    "Biography",
    "Family",
}
sub_dataframe.loc[:, 'Genre'] = sub_dataframe['Genre'].apply(lambda x: x if x not in genres_a_retirer else float('nan'))

In [400]:
sub_dataframe.value_counts('Genre')

Genre
Drama        289
Action       172
Comedy       155
Crime        107
Adventure     72
Mystery       12
Horror        11
Western        4
Fantasy        2
Thriller       1
Name: count, dtype: int64

In [401]:
def format_dataframe_by_target(dataframe,top: int=4, column: str='Genre', other_label: str="Other"):
    # garder les top N genres les plus fréquents
    top_genres = dataframe[column].value_counts().nlargest(top).index
    
    # le reste est regroupé dans une catégorie "Other" ou NaN
    dataframe.loc[:, column] = dataframe[column].apply(lambda x: x if x in top_genres else float('nan')) # else other_label) 
    
    return dataframe
    

In [402]:
sub_dataframe = format_dataframe_by_target(sub_dataframe, top=4, column='Genre')

In [403]:
# garder uniquement les genres les plus fréquents et remplacer les autres par NaN
# liste_genres_frequents = sub_dataframe['Genre'].value_counts().nlargest(4).index.tolist()
# liste_genres_frequents

# sub_dataframe.loc[:, 'Genre'] = sub_dataframe['Genre'].apply(lambda g: g if g in liste_genres_frequents else float('nan'))

In [404]:
# compter les cases vides/Nan selon les colonnes
sub_dataframe.isnull().sum()

Released_Year      1
Director           0
Certificate        0
Runtime            0
IMDB_Rating        0
Meta_score       157
No_of_Votes        0
Gross            169
Genre            277
DirectorID         0
CertificateID      0
dtype: int64

In [405]:
sub_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  999 non-null    float64
 1   Director       1000 non-null   object 
 2   Certificate    1000 non-null   object 
 3   Runtime        1000 non-null   float64
 4   IMDB_Rating    1000 non-null   float64
 5   Meta_score     843 non-null    float64
 6   No_of_Votes    1000 non-null   float64
 7   Gross          831 non-null    float64
 8   Genre          723 non-null    object 
 9   DirectorID     1000 non-null   float64
 10  CertificateID  1000 non-null   float64
dtypes: float64(8), object(3)
memory usage: 86.1+ KB


In [406]:
sub_dataframe.value_counts('Genre')

Genre
Drama     289
Action    172
Comedy    155
Crime     107
Name: count, dtype: int64

#### suppression des lignes avec des valeurs manquantes

In [407]:
output = sub_dataframe.copy()
output.dropna(inplace=True)
# output.head(10)
# output.info()
output.value_counts('Genre')

Genre
Drama     209
Action    130
Comedy    109
Crime      81
Name: count, dtype: int64

- transformer les genres minoritaires en "Other" (optionnel, à discuter) selon un seuil de fréquence défini

In [408]:
# si le nombre lignes par genre est supérieur à 1000,
# → garder le genre
# → sinon le remplacer par "Other"
# SEUIL_MINIMUM = 1000
# genre_counts = output['Genre'].value_counts()
# genres_to_keep = genre_counts[genre_counts >= SEUIL_MINIMUM].index
# output.loc[~output['Genre'].isin(genres_to_keep), 'Genre'] = "Other"

# output.value_counts('Genre')

In [409]:
sub_dataframe = output.copy()
sub_dataframe.value_counts('Genre')

Genre
Drama     209
Action    130
Comedy    109
Crime      81
Name: count, dtype: int64

- convertir les genres (**valeur prédite**) en format numérique

In [410]:
# Encodage des genres non nuls

valid_genres = sub_dataframe['Genre'].dropna().astype(str)

label_encoder.fit(valid_genres)

genre_map = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

sub_dataframe["GenreEncoded"] = sub_dataframe['Genre'].map(genre_map)
sub_dataframe["GenreEncoded"] = sub_dataframe["GenreEncoded"].astype("Int64")

In [411]:
sub_dataframe.dtypes

Released_Year    float64
Director          object
Certificate       object
Runtime          float64
IMDB_Rating      float64
Meta_score       float64
No_of_Votes      float64
Gross            float64
Genre             object
DirectorID       float64
CertificateID    float64
GenreEncoded       Int64
dtype: object

In [412]:
print("Nombre de lignes par genre encodé :")
print(sub_dataframe[['Genre', 'GenreEncoded']].value_counts())

Nombre de lignes par genre encodé :
Genre   GenreEncoded
Drama   3               209
Action  0               130
Comedy  1               109
Crime   2                81
Name: count, dtype: int64


In [413]:
sub_dataframe.info()

<class 'pandas.core.frame.DataFrame'>
Index: 529 entries, 0 to 997
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Released_Year  529 non-null    float64
 1   Director       529 non-null    object 
 2   Certificate    529 non-null    object 
 3   Runtime        529 non-null    float64
 4   IMDB_Rating    529 non-null    float64
 5   Meta_score     529 non-null    float64
 6   No_of_Votes    529 non-null    float64
 7   Gross          529 non-null    float64
 8   Genre          529 non-null    object 
 9   DirectorID     529 non-null    float64
 10  CertificateID  529 non-null    float64
 11  GenreEncoded   529 non-null    Int64  
dtypes: Int64(1), float64(8), object(3)
memory usage: 54.2+ KB


In [414]:
sub_dataframe['Genre'].value_counts()

Genre
Drama     209
Action    130
Comedy    109
Crime      81
Name: count, dtype: int64

In [415]:
sub_dataframe.isnull().sum()

Released_Year    0
Director         0
Certificate      0
Runtime          0
IMDB_Rating      0
Meta_score       0
No_of_Votes      0
Gross            0
Genre            0
DirectorID       0
CertificateID    0
GenreEncoded     0
dtype: int64

#### corrélation entre les variables cibles

In [416]:
correlation_data = sub_dataframe.drop(columns=['Genre', 'GenreEncoded', 'Director', 'Certificate']).corr()
correlation_data

,Released_Year,Runtime,IMDB_Rating,Meta_score,No_of_Votes,Gross,DirectorID,CertificateID
Released_Year,1.000000,0.075055,-0.182432,-0.326667,0.259779,0.175354,-0.016671,-0.228086
Runtime,0.075055,1.000000,0.267235,-0.002139,0.193500,0.235217,-0.034089,0.090252
IMDB_Rating,-0.182432,0.267235,1.000000,0.253581,0.446124,0.062407,-0.048502,0.098438
Meta_score,-0.326667,-0.002139,0.253581,1.000000,-0.065284,-0.060749,-0.065556,0.084862
No_of_Votes,0.259779,0.193500,0.446124,-0.065284,1.000000,0.617279,-0.044461,0.037513
Gross,0.175354,0.235217,0.062407,-0.060749,0.617279,1.000000,0.030271,0.096527
DirectorID,-0.016671,-0.034089,-0.048502,-0.065556,-0.044461,0.030271,1.000000,-0.032609
CertificateID,-0.228086,0.090252,0.098438,0.084862,0.037513,0.096527,-0.032609,1.000000


In [417]:
sub_dataframe.value_counts('Genre')

Genre
Drama     209
Action    130
Comedy    109
Crime      81
Name: count, dtype: int64

### exporter correspondance entre colonnes initiales et valeurs numériques dans un fichier JSON
- genres, directeurs...

In [418]:
def export_encoding_map(df, col_originale, col_encodee, output_path):

    mapping_df = df[[col_originale, col_encodee]].drop_duplicates().sort_values(col_encodee)
    
    # Déterminer les types réels
    is_originale_numeric = pd.api.types.is_numeric_dtype(df[col_originale])
    is_encodee_numeric = pd.api.types.is_numeric_dtype(df[col_encodee])
    
    # Construire le mapping avec conversions appropriées
    mapping = {}
    for _, row in mapping_df.iterrows():
        # Clé: convertir en string si numérique, sinon garder comme string
        key = str(int(row[col_originale])) if is_originale_numeric else str(row[col_originale])
        
        # Valeur: convertir en int si numérique, sinon garder comme string
        value = int(row[col_encodee]) if is_encodee_numeric else str(row[col_encodee])
        
        mapping[key] = value
    
    os.makedirs(os.path.dirname(output_path) or '.', exist_ok=True)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(mapping, f, indent=4, ensure_ascii=False)
    
    print(f"Mapping exporté vers {output_path} :")
    print(json.dumps(mapping, indent=4, ensure_ascii=False))

In [419]:
# Genre original → Encoding numérique
export_encoding_map(sub_dataframe, 'Genre', 'GenreEncoded', '../data/genre_to_encoded.json')
# Encoding numérique → Genre original
export_encoding_map(sub_dataframe, 'GenreEncoded', 'Genre', '../data/encoded_to_genre.json')

# Director original → Encoding numérique
export_encoding_map(sub_dataframe, 'Director', 'DirectorID', '../data/director_to_encoded.json')
# Encoding numérique → Director original
export_encoding_map(sub_dataframe, 'DirectorID', 'Director', '../data/encoded_to_director.json')

# Certificate original → Encoding numérique
export_encoding_map(sub_dataframe, 'Certificate', 'CertificateID', '../data/certificate_to_encoded.json')
# Encoding numérique → Certificate original
export_encoding_map(sub_dataframe, 'CertificateID', 'Certificate', '../data/encoded_to_certificate.json')

Mapping exporté vers ../data/genre_to_encoded.json :
{
    "Action": 0,
    "Comedy": 1,
    "Crime": 2,
    "Drama": 3
}
Mapping exporté vers ../data/encoded_to_genre.json :
{
    "0": "Action",
    "1": "Comedy",
    "2": "Crime",
    "3": "Drama"
}
Mapping exporté vers ../data/director_to_encoded.json :
{
    "abdellatif kechiche": 2,
    "abhishek kapoor": 4,
    "akira kurosawa": 9,
    "alan parker": 12,
    "alejandro g. iñárritu": 14,
    "alex garland": 16,
    "alfonso cuarón": 20,
    "alfonso gomez-rejon": 21,
    "alfred hitchcock": 22,
    "anders thomas jensen": 24,
    "andrew davis": 27,
    "andrew lau": 28,
    "andrew niccol": 29,
    "andrey zvyagintsev": 31,
    "aneesh chaganty": 32,
    "ang lee": 33,
    "anthony russo": 36,
    "antoine fuqua": 37,
    "asghar farhadi": 42,
    "barry levinson": 44,
    "ben affleck": 45,
    "bernardo bertolucci": 47,
    "billy bob thornton": 48,
    "billy wilder": 49,
    "bob clark": 52,
    "bob fosse": 53,
    "bong joo

### Exporter le dataset en CSV

In [420]:
# sub_dataframe.drop(columns=['Genre'], inplace=True)
sub_dataframe = sub_dataframe[['Released_Year', 'Runtime', 'IMDB_Rating', 'Meta_score', 'No_of_Votes', 'Gross', 'DirectorID', 'CertificateID', 'GenreEncoded']]

In [ ]:
sub_dataframe.to_csv('../data/top_movies_cleaned.csv', index=False)
end = time.time() - start

print(f"Temps d'exécution : {end:.2f} secondes")

Temps d'exécution : 3.05 secondes


: 